# Average images

## Import modules

In [1]:
# import internal modules
from typing import List, Set, Dict, Tuple, Optional, Union, Callable
from pathlib import Path
from datetime import date
from uuid import uuid4
from random import shuffle

# import 3rd-party modules
import cv2
import numpy as np
from imageio import get_writer
from pygifsicle import optimize

## Define classes

In [ ]:
class Project:
    """
    Class to manage projects
    Arguments:
    * project_dir: string or Path object for project directory. It is required
    * out_img_dir: string or Path object f
    """
    def __init__(
        self,
        project_dir: Union[str,Path],
        in_img_dir:Optional[str] = None,
        in_img_path_list:Optional[List[str]] = None,
        out_img_dir_list: Optional[List[Union[str,Path]]] = None,
        parents: Optional[bool] = True,
        exist_ok: Optional[bool] = True,
        in_project_dir: bool = True,
        **get_img_path_args
    ) -> None:
        """
        Function to create an instance of Project class
        """
        # convert project dir to path if string provided
        self.project_dir = Path(project_dir)

        # create project directory if it doesn't exist
        self.make_dir(self.project_dir, parents=parents, exist_ok=exist_ok, in_project_dir=False, is_project_dir=True)

        self.in_img_dir = in_img_dir
        self.out_img_dir_list = out_img_dir_list
        
        # initiate output image directory dictionary
        self.out_img_dir_dict = dict()

        # if list is none, initialize it
        if in_img_path_list is None:
            self.in_img_path_list = []
        if out_img_dir_list is None:
            self.out_img_dir_list = []

        # iterate over list of output image directory
        for out_img_dir in self.out_img_dir_list:
            self.make_dir(out_img_dir, parents=parents, exist_ok=exist_ok, in_project_dir=in_project_dir)

        # if input image directory given
        if in_img_dir is not None:
            # if in project dir enabled
            if in_project_dir:
                self.in_img_dir = self.project_dir / in_img_dir
            else:
                self.in_img_dir = Path(in_img_dir)

        # get img paths list
        self.in_img_path_list = self.get_img_path_list(img_dir=self.in_img_dir, img_path_list=self.in_img_path_list, **get_img_path_args)


    def make_dir(
        self,
        dir: Union[str,Path],
        parents: Optional[bool] = True,
        exist_ok: Optional[bool] = True,
        in_project_dir: Optional[bool] = True,
        is_project_dir: Optional[bool] = False
    ) -> None:
        """
        Function to make directory
        """
        # if create in project dir enabled
        if in_project_dir:
            dir = self.project_dir / dir

        else:
            # just convert dir to path if string provided
            dir = Path(dir)
 
        # create dir if don't exist
        # parents=True to create any intermediate parent dirs if don't exist
        dir.mkdir(parents=parents, exist_ok=exist_ok)
        
        # if this function is not called to create the project dir itself
        if not is_project_dir:
            # add dir to out_img_dir_dict
            self.out_img_dir_dict[dir.name] = dir
    

    @staticmethod
    def get_img_path_list(
        img_dir:Optional[Union[str,Path]]=None,
        img_path_list:Optional[List[str]]=None,
        img_extensions:Set[str]={'.png', '.jpg', '.jpeg', '.JPG', '.JPEG', '.PNG'},
        glob_exp:str="**/*",
        sort_img_list:bool=True,
        reverse_img_list:bool=False,
    ) -> List[str]:
        """
        Utility function to get a list of img paths
        """
        # initialize img_path_list if not provided
        if img_path_list is None:
            img_path_list = []

        # if image directory provided
        if img_dir is not None:
            # convert img_dir to path if string provided
            img_dir = Path(img_dir)

            # get list of images in img directory and extend to img path list
            img_path_list.extend([str(img_path) for img_path in img_dir.glob(glob_exp) if img_path.suffix in img_extensions])

        # if sort_img_list is True, sort image paths list
        if sort_img_list:
            img_path_list = sorted(img_path_list, reverse=reverse_img_list)

        return img_path_list

## Define functions

In [ ]:
def get_interpolation(src_img_shape:Tuple[int], out_img_shape:Optional[Tuple[int]]=None, out_img_scale:Optional[Tuple[int]]=None) -> Optional[int]:
    """
    Function to get best interpolation flag for resizing based on if it is a downscale or upscale of an image
    """
    # get source and output shape
    src_img_height, src_img_width = src_img_shape[:2]

    if out_img_shape is not None:
        out_img_height, out_img_width = out_img_shape[:2]

    elif out_img_scale is not None:
        out_img_scale_fy, out_img_scale_fx = out_img_scale
        out_img_height, out_img_width = src_img_height * out_img_scale_fy, src_img_width * out_img_scale_fx

    else:
        return None

    # if input and output shapes are the same, no interpolation needed
    if (src_img_height == out_img_height) and (src_img_width == out_img_width):
        return None

    # if downscale, use INTER_AREA interpolation
    elif (src_img_height > out_img_height) or (src_img_width > out_img_width):
        return cv2.INTER_AREA

    # if upscale, use INTER_CUBIC or INTER_LINEAR interpolation
    else:
        return cv2.INTER_CUBIC

In [ ]:
def resize_with_pad(
    img:Optional[np.array] = None,
    ref_img:Optional[np.array] = None,
    img_path:Optional[str] = None,
    ref_img_path:Optional[str] = None,
    ref_img_shape:Optional[Tuple[int]] = None,
    cvt_color:Optional[int] = None
    ) -> np.array:
    """
    Function to resize image with padding to keep same aspect ratio
    """

    # read image if image path passed
    if img_path is not None:
        img = cv2.imread(img_path)

    if cvt_color is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # read ref image if ref image path passed and no image shape passed
    if (ref_img_path is not None) and (ref_img_shape is None):
        ref_img = cv2.imread(ref_img_path)
    
    if ref_img is not None:
        ref_img_shape = ref_img.shape

    img_height, img_width, img_channel = img.shape
    ref_img_height, ref_img_width, img_channel = ref_img_shape
    
    # get aspect ratio of both images
    img_ratio = img_width/img_height
    ref_img_ratio = ref_img_width/ref_img_height

    # if same aspect ratio, no need for padding
    if img_ratio == ref_img_ratio:
        out_img = img

    # else, padding required
    else:

        # if img_ratio is smaller than ref_img_ratio, add padding to width to have the same aspect ratio as reference
        if img_ratio < ref_img_ratio:
            # get new width
            new_width = round(img_height*ref_img_ratio)
            out_img = np.zeros((img_height, new_width, img_channel), dtype=img.dtype)

            # get yx positions to center image
            y, x = 0, (new_width - img_width)//2

        # if img_ratio is smaller than ref_img_ratio, add padding to height to have the same aspect ratio as reference
        else:
            # get new height
            new_height = round(img_width/ref_img_ratio)
            out_img = np.zeros((new_height, img_width, img_channel), dtype=img.dtype)

            # get yx positions to center image
            y, x = (new_height - img_height)//2, 0

        # paste image at the center of black canvas
        out_img[y:y+img_height, x:x+img_width] = img

    # get interpolation
    interpolation = get_interpolation(src_img_shape=img.shape, out_img_shape=(ref_img_height, ref_img_width, img_channel))

    # return resize image with zeros padding (to get the reference ratio) to reference shape
    return cv2.resize(out_img, (ref_img_width, ref_img_height), interpolation=interpolation)

In [ ]:
def resize_with_crop(
    img:Optional[np.array] = None,
    ref_img:Optional[np.array] = None,
    img_path:Optional[str] = None,
    ref_img_path:Optional[str] = None,
    ref_img_shape:Optional[Tuple[int]] = None,
    cvt_color:Optional[int] = None
    ) -> np.array:
    """
    Function to resize image after cropping to keep same aspect ratio
    """

    # read image if image path passed
    if img_path is not None:
        img = cv2.imread(img_path)

    if cvt_color is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

    # read ref image if ref image path passed and no image shape passed
    if (ref_img_path is not None) and (ref_img_shape is None):
        ref_img = cv2.imread(ref_img_path)
    
    if ref_img is not None:
        ref_img_shape = ref_img.shape

    # get images shapes
    img_height, img_width, img_channel = img.shape
    ref_img_height, ref_img_width, img_channel = ref_img_shape
    
    # get aspect ratio of both images
    img_ratio = img_width/img_height
    ref_img_ratio = ref_img_width/ref_img_height

    # if same aspect ratio, no need for cropping
    if img_ratio == ref_img_ratio:
        out_img = img.copy()

    # else, cropping required
    else:

        # if img_ratio is smaller than ref_img_ratio, crop width to have the same aspect ratio as reference
        if img_ratio > ref_img_ratio:
            # get new width
            new_width = round(img_height*ref_img_ratio)

            # get yx positions to center image
            y, x = 0, (img_width - new_width)//2

            # crop image from center
            out_img = img[y:y+img_height, x:x+new_width]

        # if img_ratio is bigger than ref_img_ratio, crop height to have the same aspect ratio as reference
        else:
            # get new height
            new_height = round(img_width/ref_img_ratio)

            # get yx positions to center image
            y, x = (img_height - new_height)//2, 0

            # crop image from center
            out_img = img[y:y+new_height, x:x+img_width]

    # get interpolation
    interpolation = get_interpolation(src_img_shape=img.shape, out_img_shape=(ref_img_height, ref_img_width, img_channel))

    # return resize image (after cropping to get the reference ratio) to reference shape
    return cv2.resize(out_img, (ref_img_width, ref_img_height), interpolation=interpolation)

In [ ]:
def create_gif(
    img_dir:Optional[str]=None,
    img_path_list:List[str]=None,
    out_path:Optional[str]=None,
    img_extensions:Set[str]={'.png', '.jpg', '.jpeg'},
    glob_exp:str="**/*",
    sort_img_list:bool=True,
    reverse_img_list:bool=False,
    shuffle_img_list:bool=False,
    writer_mode:str='I',
    duplicate_start_img_amount:int=0,
    duplicate_end_img_amount:int=0,
    out_img_shape:Optional[Tuple[int]]=None,
    resize_fct:Optional[Callable]=None,
    optimize_gif:Optional[bool]=True
    ):
    """
    Program to create animated animated gif from images in given directory and/or subdirectory
    or list

    Arguments:
    * img_dir: images directory
    * out_path: path where to save the output gif; if None, save gif with a random unique identifier
    * img_extensions: set of extensions to look for in images directory
    * reverse_img_list: boolean to reverse list of images; False by default
    * optimize_gif: to reduce gif memory sif (caution: can decrease the quality)
    """
    # if list of image paths is not given
    if img_path_list is None:
        # initialize list
        img_path_list = []
        
    # if image directory provided
    if img_dir is not None:
        # convert img_dir to Path
        img_dir = Path(img_dir)

        # get list of images in img directory and extend to img path list
        img_path_list.extend([str(img_path) for img_path in img_dir.glob(glob_exp) if img_path.suffix in img_extensions])

    # if sort_img_list is true and shuffle_img_list false, sort image paths list
    if sort_img_list and not shuffle_img_list:
        img_path_list = sorted(img_path_list, reverse=reverse_img_list)

    # if shuffle_img_list is true, shuffle img path list
    elif shuffle_img_list:
        shuffle(img_path_list)

    # if output path is not given, set gif filename with a random unique identifier
    if out_path is None:
        # generate a random uuid and convert it to string
        out_path = f"{uuid4()}.gif"

    # write gif within context manager
    with get_writer(out_path, mode=writer_mode) as writer:

        # get number of image paths
        nb_imgs = len(img_path_list)

        # if out_img_shape is provided, unpack it
        if out_img_shape is not None:
            out_img_height, out_img_width = out_img_shape[:2]
        else:
            out_img_height, out_img_width = (None, None)

        # iterate over the images to add frame to gif
        for img_nb, img_path in enumerate(img_path_list, start=1):
            img = cv2.imread(img_path)

            source_img_height, source_img_width = img.shape[:2]

            if out_img_shape is not None:
                
                if resize_fct is not None:
                    img = resize_fct(img=img, ref_img_shape=out_img_shape)

                else:

                    interpolation = get_interpolation((source_img_height, source_img_width), out_img_shape=out_img_shape)
                    
                    # if needed, resize image
                    if interpolation is not None:
                        img = cv2.resize(img, (out_img_width, out_img_height), interpolation=interpolation)

            # convert image to RGB
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # write several frames for beginning and ending
            if img_nb == 1:
                for _ in range(duplicate_start_img_amount):
                    writer.append_data(img)
            
            if img_nb == nb_imgs:
                for _ in range(duplicate_end_img_amount):
                    writer.append_data(img)

            # write output frame
            writer.append_data(img)

    if optimize_gif:
        # optimize gif to reduce size
        optimize(out_path)

In [ ]:
def create_video(
    img_dir:Optional[str]=None,
    img_path_list:List[str]=None,
    out_path:Optional[str]=None,
    codec='MP4V',
    fps=25,
    img_extensions:Set[str]={'.png', '.jpg', '.jpeg'},
    glob_exp:str="**/*",
    sort_img_list:bool=True,
    reverse_img_list:bool=False,
    shuffle_img_list:bool=False,
    duplicate_start_img_amount:int=0,
    duplicate_end_img_amount:int=0,
    out_img_shape:Optional[Tuple[int]]=None,
    resize_fct:Optional[Callable]=None,
    rotate_90:Optional[int]=None,
    ):
    """
    Function to create video using list of saved images in given directory and/or subdirectory
    or list
    """
    # if list of image paths is not given
    if img_path_list is None:
        # initialize list
        img_path_list = []

    if img_dir is not None:
        # convert img_dir to Path
        img_dir = Path(img_dir)

        # get list of images in img directory and extend to img path list
        img_path_list.extend([str(img_path) for img_path in img_dir.glob(glob_exp) if img_path.suffix in img_extensions])
        
    # if sort_img_list is true and shuffle_img_list false, sort image paths list
    if sort_img_list and not shuffle_img_list:
        img_path_list = sorted(img_path_list, reverse=reverse_img_list)

    # if shuffle_img_list is true, shuffle img path list
    elif shuffle_img_list:
        shuffle(img_path_list)

    # get number of image paths
    nb_imgs = len(img_path_list)

    # get shape of first image in list (if no output shape provided, the shape of first image will be the output shape)
    img_height, img_width, img_channel = cv2.imread(img_path_list[0]).shape

    interpolation = get_interpolation((img_height, img_width), out_img_shape=out_img_shape)
    
    # if resize needed, update img_height, img_width
    if interpolation is not None:
        img_height, img_width, img_channel = out_img_shape

    # if output path is not given, set gif filename with a random unique identifier
    if out_path is None:
        # generate a random uuid and convert it to string
        out_path = f"{uuid4()}.{codec[:-1]}"
    

    # create a videoWriter object
    fourcc = cv2.VideoWriter_fourcc(*codec)
    out_video = cv2.VideoWriter(filename=out_path, fourcc=fourcc, fps=fps, frameSize=(img_width, img_height))

    # iterate over the images to add frame to gif
    for img_nb, img_path in enumerate(img_path_list, start=1):
        img = cv2.imread(img_path)

        if rotate_90 is not None:
            img = np.rot90(img, rotate_90)

        if resize_fct is not None:
            img = resize_fct(img=img, ref_img_shape=(img_height, img_width, img_channel))

        else:
            interpolation = get_interpolation((img_height, img_width), out_img_shape=img.shape[:2])

            # if needed, resize image
            if interpolation is not None:
                img = cv2.resize(img, (img_width, img_height), interpolation=interpolation)

        # write several frames for beginning and ending
        if img_nb == 1:
            for _ in range(duplicate_start_img_amount):
                out_video.write(img)
        
        if img_nb == nb_imgs:
            for _ in range(duplicate_end_img_amount):
                out_video.write(img)

        # write output frame
        out_video.write(img)

    # release video rendering
    out_video.release()

## Set up project

In [ ]:
# name out img dirs
out_img_dir_list = ["out"]

# create project
project = Project(project_dir="assets/images/average_images", out_img_dir_list=out_img_dir_list)

## Read images

In [ ]:
# set img_path_list
img_path_list = project.get_img_path_list(img_dir="/Users/derrickvanfrausum/BeCode_AI/git-repos/coding-art/core/assets/images/morph/db_split")

In [ ]:
out_img_dir = "atomium_average"
project.make_dir(out_img_dir)

In [ ]:
# create empty list of frames
imgs = []

# read first image to get a reference shape
ref_img = cv2.imread(img_path_list[0])
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape

imgs.append(ref_img)

# iterate over image paths (except first one in the list as it's already read)
for img_path in img_path_list[1:]:

    # resize while keeping same aspect ratio (thanks to padding or cropping)
    img = resize_with_crop(img_path=img_path, ref_img_shape=(ref_img_height, ref_img_width, ref_img_channel))

    # append frame to list of frames
    imgs.append(img)

# Average images

In [ ]:
# get current date
today = date.today().strftime("%Y%m%d")

np.random.shuffle(imgs)

# compute average of frames
out_img = np.median(imgs[:5], axis=0).astype(dtype=np.uint8)

# set output image path
out_img_path = str(project.project_dir / f"average_atomium_{today}.jpg")

# save output image
cv2.imwrite(out_img_path, out_img)

## Create gif or video from images used to average

In [ ]:
# read first image to get a reference shape
ref_img = cv2.imread(img_path_list[0])
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape

# set video path
video_path = str(project.project_dir / f"{out_img_dir_list[0]}/crop_{today}_h264.mp4")

# create video
create_video(img_path_list=img_path_list, out_path=video_path, fps=12, codec='H264',
sort_img_list=True, duplicate_start_img_amount=0, duplicate_end_img_amount=0,
resize_fct=resize_with_crop, out_img_shape=(ref_img_height//2, ref_img_width//2, ref_img_channel))

# set gif path
gif_path = str(project.project_dir / f"{out_img_dir_list[0]}/crop_{today}.gif")

# create video
create_gif(img_path_list=img_path_list, out_path=gif_path,
sort_img_list=True, duplicate_start_img_amount=0, duplicate_end_img_amount=0,
resize_fct=resize_with_crop, out_img_shape=(int(ref_img_height//5.2), int(ref_img_width//5.2), ref_img_channel))

## Average video

In [ ]:
out_img_dir = "3d_origami"
project.make_dir(out_img_dir)

# set number of times to generate average
nb_averaging = 1

for i in range(nb_averaging):

    # set video path
    filepath = Path("assets/3d/renders/tuto_origami_anim0001-0300.mp4")

    # initialize video stream
    cap = cv2.VideoCapture(str(filepath))

    fps = cap.get(cv2.CAP_PROP_FPS)

    # get number of frames
    nb_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    # set number of frames to consider
    nb_frames_to_average = 10

    # get frame indexes
    frame_indexes = np.random.randint(0, nb_frames - 1, nb_frames_to_average)

    # create empty list of frames
    frames = []

    # iterate over frame indexes
    for frame_idx in frame_indexes:
        # set frame position to the index
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

        # read frame
        ret, frame = cap.read()

        if ret:

            # append frame to list of frames
            frames.append(frame)

    # compute average of frames*
    bg_frame = np.median(frames, axis=0).astype(dtype=np.uint8)

    # save image
    today = date.today().strftime("%Y%m%d")

    # set output image path
    bg_frame_path = f"assets/images/average_images/{out_img_dir}/{filepath.stem}_{today}_random_{nb_frames_to_average}_{uuid4()}.jpg"

    # save output image
    cv2.imwrite(bg_frame_path, bg_frame)

    # release video stream
    cap.release()

## Average every n frames in video

In [ ]:
# get current date
today = date.today().strftime("%Y%m%d")

# set video path
filepath = Path("/Users/derrickvanfrausum/Desktop/danse_gagu.mp4")
out_path = f"assets/images/average_images/{filepath.stem}_{today}.mp4"

codec = "H264"

# initialize video stream
cap = cv2.VideoCapture(str(filepath))

frame_factor = 100

fps = cap.get(cv2.CAP_PROP_FRAME_COUNT)/frame_factor

_, first_frame = cap.read()
ref_img_height, ref_img_width, ref_img_channel = first_frame.shape

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=out_path, fourcc=fourcc, fps=fps, frameSize=(ref_img_width, ref_img_height))

# create empty list of frames
frames = []
frame_nb = 0

while True:

    # read frame
    ret, frame = cap.read()

    if not ret:
        break

    # append frame to list of frames
    frames.append(frame)

    if frame_nb%frame_factor == 0:
        # compute average of frames
        out_frame = np.median(frames, axis=0).astype(dtype=np.uint8)

        # write output frame
        out_video.write(out_frame)

        frames = []

    frame_nb += 1

# release video stream & video rendering
cap.release()
out_video.release()